In [ ]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [ ]:
CHECKPOINT_DIR = Path("notebooks/models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "leaderboard_week2_best.pt"
print("Checkpoint will be saved to:", CHECKPOINT_PATH)

In [ ]:
# Standard library
import copy
import inspect
import json
import random

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Scikit-learn
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Neuroprobe
import neuroprobe
import neuroprobe.train_test_splits as neuroprobe_train_test_splits
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [ ]:
# =========================
# Config
# =========================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASKS = ["delta_volume", "speech", "pitch", "gpt2_surprisal", "word_gap"]

BATCH_SIZE = 8
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

STFT_N_FFT = 64
STFT_HOP = 16
STFT_WIN_LEN = 32
LAPLACIAN_K = 4

SUBJ_EMB_DIM = 16
ELEC_EMB_DIM = 16
TASK_EMB_DIM = 8

TEST_SUBJECT_ID = 1
TEST_TRIAL_ID = 2
SUBJECT_IDS = list(range(1, 11))

USE_GLOBAL_TRAIN_NORM = False
USE_VAL_AS_TEST = False

print("DEVICE:", DEVICE)

In [ ]:
subject = BrainTreebankSubject(
    subject_id=TEST_SUBJECT_ID,
    cache=True,
    dtype=torch.float32,
    coordinates_type="cortical",
)

def build_all_subjects_dict(subject_ids, test_subject):
    all_subjects = {}
    test_sid = int(test_subject.subject_id)

    for sid in map(int, subject_ids):
        if sid == test_sid:
            all_subjects[sid] = test_subject
        else:
            all_subjects[sid] = BrainTreebankSubject(
                subject_id=sid,
                cache=True,
                dtype=torch.float32,
                coordinates_type="cortical",
            )

    return all_subjects

ALL_SUBJECTS = {
    sid: (
        subject if ("subject" in globals() and int(subject.subject_id) == sid)
        else BrainTreebankSubject(
            subject_id=sid,
            cache=True,
            dtype=torch.float32,
            coordinates_type="cortical",
        )
    )
    for sid in SUBJECT_IDS
}

In [ ]:
# =========================
# Helpers
# =========================
def task_to_id_map(tasks):
    return {t: i for i, t in enumerate(tasks)}

TASK_TO_ID = task_to_id_map(TASKS)

# =========================
# Repro
# =========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# =========================
# Split helpers
# =========================
def _call_split_function(eval_name, split_idx=0):
    kwargs = {
        "eval_name": eval_name,
        "all_subjects": ALL_SUBJECTS,
        "test_subject_id": TEST_SUBJECT_ID,
        "test_trial_id": TEST_TRIAL_ID,
        "dtype": torch.float32,
        "lite": True,
        "nano": False,
    }

    print(f"Calling generate_splits_cross_subject with kwargs: {kwargs}")
    splits = neuroprobe_train_test_splits.generate_splits_cross_subject(**kwargs)

    fold = splits[split_idx] if isinstance(splits, list) else splits

    train_ds = fold["train_dataset"]

    if USE_VAL_AS_TEST and "val_dataset" in fold:
        eval_ds = fold["val_dataset"]
    elif "test_dataset" in fold:
        eval_ds = fold["test_dataset"]
    elif "val_dataset" in fold:
        eval_ds = fold["val_dataset"]
    else:
        raise KeyError(
            f"Expected one of 'test_dataset' or 'val_dataset' in fold keys, got: {list(fold.keys())}"
        )

    return train_ds, eval_ds, fold

def get_cross_subject_datasets(task_name, split_idx=0):
    return _call_split_function(eval_name=task_name, split_idx=split_idx)

# =========================
# Laplacian helper
# =========================
def build_laplacian_matrix(coords, k_neighbors=4):
    coords = np.asarray(coords, dtype=np.float32)
    n_electrodes = coords.shape[0]

    nn_model = NearestNeighbors(
        n_neighbors=min(k_neighbors + 1, n_electrodes),
        metric="euclidean",
    )
    nn_model.fit(coords)
    distances, indices = nn_model.kneighbors(coords)

    W = np.zeros((n_electrodes, n_electrodes), dtype=np.float32)

    for e in range(n_electrodes):
        neigh = [j for j in indices[e] if j != e]
        if len(neigh) == 0:
            continue

        neigh_d = []
        for j in neigh:
            d = np.linalg.norm(coords[e] - coords[j])
            neigh_d.append(max(d, 1e-6))
        neigh_d = np.asarray(neigh_d, dtype=np.float32)

        weights = 1.0 / neigh_d
        weights = weights / weights.sum()

        for w, j in zip(weights, neigh):
            W[e, j] = w

    return W

# =========================
# Dataset unpack helpers
# =========================
def unpack_base_item(item):
    if isinstance(item, dict):
        x = item["data"]
        y = item["label"]
        meta = item
    else:
        x, y = item[0], item[1]
        meta = {}

    if not torch.is_tensor(x):
        x = torch.tensor(x, dtype=torch.float32)
    else:
        x = x.to(torch.float32)

    y = torch.tensor(float(y), dtype=torch.float32)
    return x, y, meta

def infer_subject_id(meta):
    if "metadata" in meta and isinstance(meta["metadata"], dict):
        md = meta["metadata"]
        for key in ["subject_id", "subject_idx", "subject", "patient_id", "participant_id"]:
            if key in md:
                try:
                    return int(md[key])
                except (TypeError, ValueError):
                    pass

    for key in ["subject_id", "subject_idx", "subject", "patient_id", "participant_id"]:
        if key in meta:
            try:
                return int(meta[key])
            except (TypeError, ValueError):
                    pass

    raise KeyError(f"Could not infer subject_id from metadata keys: {list(meta.keys())}")

def infer_electrode_coordinates(meta, fallback_coords):
    if "metadata" in meta and isinstance(meta["metadata"], dict):
        md = meta["metadata"]
        for key in ["electrode_coordinates", "coords", "coordinates"]:
            if key in md:
                arr = np.asarray(md[key], dtype=np.float32)
                if arr.ndim == 2 and arr.shape[1] == 3:
                    return arr

    for key in ["electrode_coordinates", "coords", "coordinates"]:
        if key in meta:
            arr = np.asarray(meta[key], dtype=np.float32)
            if arr.ndim == 2 and arr.shape[1] == 3:
                return arr
    return fallback_coords

# =========================
# Global spectrogram stats
# =========================
@torch.no_grad()
def estimate_train_spec_stats(
    base_ds,
    fallback_coords,
    lap_k=4,
    n_fft=64,
    hop_length=16,
    win_length=32,
    max_samples=256,
):
    fallback_coords = np.asarray(fallback_coords, dtype=np.float32)

    sum_vec = None
    sumsq_vec = None
    count_vec = None

    window = torch.hann_window(win_length)

    n = min(len(base_ds), max_samples)
    if n == 0:
        raise ValueError("Empty training dataset when estimating spectrogram stats.")

    sample_indices = np.linspace(0, len(base_ds) - 1, n, dtype=int)

    for idx in sample_indices:
        x, _, meta = unpack_base_item(base_ds[idx])
        coords = infer_electrode_coordinates(meta, fallback_coords)

        lap_w = torch.tensor(
            build_laplacian_matrix(coords, lap_k),
            dtype=torch.float32,
            device=x.device,
        )
        x_lap = x - lap_w @ x

        stft = torch.stft(
            x_lap,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            window=window.to(dtype=x_lap.dtype, device=x_lap.device),
            center=True,
            pad_mode="reflect",
            normalized=False,
            onesided=True,
            return_complex=True,
        )

        mag = torch.log1p(torch.abs(stft))
        flat = mag.reshape(mag.shape[0], -1).double().cpu()

        if sum_vec is None:
            n_electrodes = flat.shape[0]
            sum_vec = torch.zeros(n_electrodes, dtype=torch.float64)
            sumsq_vec = torch.zeros(n_electrodes, dtype=torch.float64)
            count_vec = torch.zeros(n_electrodes, dtype=torch.float64)

        if flat.shape[0] != sum_vec.shape[0]:
            raise ValueError(
                f"Inconsistent electrode count across samples in estimate_train_spec_stats: "
                f"expected {sum_vec.shape[0]}, got {flat.shape[0]} at idx={idx}"
            )

        sum_vec += flat.sum(dim=1)
        sumsq_vec += (flat ** 2).sum(dim=1)
        count_vec += flat.shape[1]

    mean = sum_vec / count_vec.clamp_min(1.0)
    var = (sumsq_vec / count_vec.clamp_min(1.0)) - mean ** 2
    std = torch.sqrt(var.clamp_min(1e-6))

    return mean.float(), std.float()


In [7]:
# =========================
# Lazy dataset
# =========================
class CrossSubjectLaplacianSpectrogramDataset(Dataset):
    def __init__(
        self,
        base_ds,
        fallback_coords,
        n_fft=64,
        hop_length=16,
        win_length=32,
        lap_k=4,
        global_mean=None,
        global_std=None,
    ):
        self.base_ds = base_ds
        self.fallback_coords = np.asarray(fallback_coords, dtype=np.float32)
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.lap_k = lap_k
        self.window = torch.hann_window(win_length)
        self.global_mean = global_mean
        self.global_std = global_std

    def __len__(self):
        return len(self.base_ds)

    def _laplacian_reference(self, x, coords):
        lap_w = torch.tensor(
            build_laplacian_matrix(coords, self.lap_k),
            dtype=x.dtype,
            device=x.device,
        )
        return x - lap_w @ x

    def _spectrogram(self, x_lap):
        window = self.window.to(device=x_lap.device, dtype=x_lap.dtype)

        stft = torch.stft(
            x_lap,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=window,
            center=True,
            pad_mode="reflect",
            normalized=False,
            onesided=True,
            return_complex=True,
        )

        mag = torch.log1p(torch.abs(stft))
        flat = mag.reshape(mag.shape[0], -1)

        if self.global_mean is not None and self.global_std is not None:
            if self.global_mean.shape[0] == flat.shape[0]:
                mean = self.global_mean.to(flat.device).unsqueeze(1)
                std = self.global_std.to(flat.device).unsqueeze(1).clamp_min(1e-6)
            else:
                mean = flat.mean(dim=1, keepdim=True)
                std = flat.std(dim=1, keepdim=True).clamp_min(1e-6)
        else:
            mean = flat.mean(dim=1, keepdim=True)
            std = flat.std(dim=1, keepdim=True).clamp_min(1e-6)

        return ((flat - mean) / std).reshape_as(mag)

    def __getitem__(self, idx):
        x, y, meta = unpack_base_item(self.base_ds[idx])

        subject_id = infer_subject_id(meta)
        coords = infer_electrode_coordinates(meta, self.fallback_coords)

        if coords.shape[0] != x.shape[0]:
            raise ValueError(
                f"Electrode coordinate count ({coords.shape[0]}) does not match signal electrode count ({x.shape[0]}) "
                f"for idx={idx}"
            )

        x_lap = self._laplacian_reference(x, coords)
        x_spec = self._spectrogram(x_lap)

        return {
            "x_spec": x_spec,  # [E, F, T]
            "y": y,
            "subject_idx": torch.tensor(subject_id, dtype=torch.long),
            "coords": torch.as_tensor(coords, dtype=torch.float32),  # [E, 3]
            "n_electrodes": torch.tensor(x_spec.shape[0], dtype=torch.long),
        }


def collate_cross_subject_batch(batch):
    max_e = max(item["x_spec"].shape[0] for item in batch)

    x_specs = []
    coords_list = []
    electrode_mask = []
    ys = []
    subject_idxs = []
    n_electrodes = []

    for item in batch:
        x_spec = item["x_spec"]
        coords = item["coords"]
        e, f, t = x_spec.shape
        pad_e = max_e - e

        if pad_e > 0:
            x_pad = torch.zeros((pad_e, f, t), dtype=x_spec.dtype)
            c_pad = torch.zeros((pad_e, 3), dtype=coords.dtype)
            m_pad = torch.zeros(pad_e, dtype=torch.bool)

            x_spec = torch.cat([x_spec, x_pad], dim=0)
            coords = torch.cat([coords, c_pad], dim=0)
            mask = torch.cat([torch.ones(e, dtype=torch.bool), m_pad], dim=0)
        else:
            mask = torch.ones(e, dtype=torch.bool)

        x_specs.append(x_spec)
        coords_list.append(coords)
        electrode_mask.append(mask)
        ys.append(item["y"])
        subject_idxs.append(item["subject_idx"])
        n_electrodes.append(item["n_electrodes"])

    return {
        "x_spec": torch.stack(x_specs, dim=0),           # [B, Emax, F, T]
        "coords": torch.stack(coords_list, dim=0),       # [B, Emax, 3]
        "electrode_mask": torch.stack(electrode_mask, 0),# [B, Emax]
        "y": torch.stack(ys, dim=0),
        "subject_idx": torch.stack(subject_idxs, dim=0),
        "n_electrodes": torch.stack(n_electrodes, dim=0),
    }

def get_fallback_coords(
    all_subjects=ALL_SUBJECTS,
    test_subject_id=TEST_SUBJECT_ID,
    trial_id: int = TEST_TRIAL_ID,
    eval_name: str = "gpt2_surprisal",
):
    base_subject = all_subjects[test_subject_id]

    ds = BrainTreebankSubjectTrialBenchmarkDataset(
        base_subject,
        trial_id=trial_id,
        dtype=torch.float32,
        eval_name=eval_name,
        lite=True,
    )

    coords = np.asarray(ds.electrode_coordinates, dtype=np.float32)
    print(
        f"Using fallback_coords from subject={test_subject_id}, "
        f"trial={trial_id}, eval_name='{eval_name}', shape={coords.shape}"
    )
    return coords


In [8]:
# =========================
# Model blocks
# =========================
class FiLM2d(nn.Module):
    def __init__(self, cond_dim, num_channels):
        super().__init__()
        self.to_gamma = nn.Linear(cond_dim, num_channels)
        self.to_beta = nn.Linear(cond_dim, num_channels)

    def forward(self, x, cond):
        gamma = self.to_gamma(cond).unsqueeze(-1).unsqueeze(-1)
        beta = self.to_beta(cond).unsqueeze(-1).unsqueeze(-1)
        return x * (1.0 + gamma) + beta


class ResidualConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)

        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.relu(x + residual)
        return x


class LeaderboardSpectrogramEncoder(nn.Module):
    def __init__(self, num_subjects, task_count, dropout=0.25, max_electrodes=512):
        super().__init__()

        self.subject_embedding = nn.Embedding(num_subjects, SUBJ_EMB_DIM)
        self.electrode_embedding = nn.Embedding(max_electrodes, ELEC_EMB_DIM)
        self.task_embedding = nn.Embedding(task_count, TASK_EMB_DIM)

        self.coord_mlp = nn.Sequential(
            nn.Linear(3, 16),
            nn.LayerNorm(16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )

        self.electrode_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualConvBlock(32, 64, stride=2),
            ResidualConvBlock(64, 96, stride=2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        cond_dim = SUBJ_EMB_DIM + ELEC_EMB_DIM + TASK_EMB_DIM + 16
        self.proj = nn.Sequential(
            nn.Linear(96 + cond_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x_spec, coords, electrode_mask, subject_idx, task_idx):
        """
        x_spec: [B, Emax, F, T]
        coords: [B, Emax, 3]
        electrode_mask: [B, Emax] bool
        subject_idx: [B]
        task_idx: [B]
        """
        batch_size, e_max, f_bins, t_bins = x_spec.shape
        device = x_spec.device

        if e_max > self.electrode_embedding.num_embeddings:
            raise ValueError(
                f"Batch has {e_max} electrodes, but encoder max_electrodes="
                f"{self.electrode_embedding.num_embeddings}"
            )

        x_flat = x_spec.reshape(batch_size * e_max, 1, f_bins, t_bins)
        spec_feat = self.electrode_cnn(x_flat).reshape(batch_size, e_max, -1)

        coord_feat = self.coord_mlp(coords.reshape(batch_size * e_max, 3)).reshape(batch_size, e_max, -1)

        elec_ids = torch.arange(e_max, device=device).unsqueeze(0).expand(batch_size, -1)
        elec_feat = self.electrode_embedding(elec_ids)

        subj_feat = self.subject_embedding(
            subject_idx.clamp(0, self.subject_embedding.num_embeddings - 1)
        )
        task_feat = self.task_embedding(task_idx)

        subj_feat = subj_feat.unsqueeze(1).expand(batch_size, e_max, -1)
        task_feat = task_feat.unsqueeze(1).expand(batch_size, e_max, -1)

        cond = torch.cat([subj_feat, elec_feat, task_feat, coord_feat], dim=-1)
        elec_repr = torch.cat([spec_feat, cond], dim=-1)

        mask = electrode_mask.unsqueeze(-1).to(elec_repr.dtype)
        elec_repr = elec_repr * mask

        pooled = elec_repr.sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        z = self.proj(pooled)
        return z


class LeaderboardMultiTaskModel(nn.Module):
    def __init__(self, num_subjects, tasks, dropout=0.25, max_electrodes=512):
        super().__init__()
        self.tasks = tasks
        self.encoder = LeaderboardSpectrogramEncoder(
            num_subjects=num_subjects,
            task_count=len(tasks),
            dropout=dropout,
            max_electrodes=max_electrodes,
        )
        self.heads = nn.ModuleDict({
            task: nn.Sequential(
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1),
            )
            for task in tasks
        })

        self.log_vars = nn.ParameterDict({
            task: nn.Parameter(torch.zeros(())) for task in tasks
        })

    def forward(self, batch, task_name):
        x_spec = batch["x_spec"]
        coords = batch["coords"]
        electrode_mask = batch["electrode_mask"]
        subject_idx = batch["subject_idx"]

        task_idx = torch.full(
            (x_spec.shape[0],),
            fill_value=TASK_TO_ID[task_name],
            device=x_spec.device,
            dtype=torch.long,
        )

        z = self.encoder(
            x_spec=x_spec,
            coords=coords,
            electrode_mask=electrode_mask,
            subject_idx=subject_idx,
            task_idx=task_idx,
        )
        pred = self.heads[task_name](z).squeeze(-1)
        return pred

    def weighted_loss(self, raw_loss, task_name):
        log_var = self.log_vars[task_name]
        return torch.exp(-log_var) * raw_loss + log_var

In [9]:
# =========================
# Metrics
# =========================
@torch.no_grad()
def evaluate_task(model, loader, criterion, device, task_name):
    model.eval()
    total_loss = 0.0
    all_y, all_preds = [], []

    for batch in loader:
        # move all batch tensors to device
        for key, value in batch.items():
            batch[key] = value.to(device)

        y = batch["y"]
        preds = model(batch, task_name)
        loss = criterion(preds, y)

        total_loss += loss.item() * y.size(0)
        all_y.append(y.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)

    metric = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) >= 2 else np.nan

    return {
        "loss": total_loss / len(loader.dataset),
        "metric_name": "auroc",
        "metric": metric,
        "baseline_metric": 0.5,
    }


def combined_score(epoch_eval):
    vals = []
    for task_name, out in epoch_eval.items():
        if np.isnan(out["metric"]):
            continue
        vals.append(out["metric"] - out["baseline_metric"])
    return float(np.mean(vals)) if vals else -np.inf

In [10]:
# =========================
# Build datasets/loaders
# =========================

fallback_coords = get_fallback_coords(
    all_subjects=ALL_SUBJECTS,
    test_subject_id=TEST_SUBJECT_ID,
    trial_id=TEST_TRIAL_ID,
)

print("fallback_coords shape:", fallback_coords.shape)

train_loaders = {}
val_loaders = {}
test_loaders = {}
train_subject_ids = set()

for task_name in TASKS:
    print(f"Preparing cross-subject loaders for {task_name}...")
    train_ds_base, test_ds_base, fold_info = get_cross_subject_datasets(task_name, split_idx=0)
    print("Fold keys:", list(fold_info.keys()))

    val_ds_base = fold_info.get("val_dataset", None)
    test_ds_base = fold_info.get("test_dataset", test_ds_base)

    global_mean, global_std = (None, None)
    if USE_GLOBAL_TRAIN_NORM:
        try:
            global_mean, global_std = estimate_train_spec_stats(
                train_ds_base,
                fallback_coords=fallback_coords,
                lap_k=LAPLACIAN_K,
                n_fft=STFT_N_FFT,
                hop_length=STFT_HOP,
                win_length=STFT_WIN_LEN,
                max_samples=256,
            )
            print(f"Estimated global train spec stats for {task_name}")
        except Exception as e:
            print(f"Global stat estimation failed for {task_name}, falling back to per-sample norm: {e}")
            global_mean, global_std = (None, None)

    train_ds = CrossSubjectLaplacianSpectrogramDataset(
        train_ds_base,
        fallback_coords=fallback_coords,
        n_fft=STFT_N_FFT,
        hop_length=STFT_HOP,
        win_length=STFT_WIN_LEN,
        lap_k=LAPLACIAN_K,
        global_mean=global_mean,
        global_std=global_std,
    )

    val_ds = (
        CrossSubjectLaplacianSpectrogramDataset(
            val_ds_base,
            fallback_coords=fallback_coords,
            n_fft=STFT_N_FFT,
            hop_length=STFT_HOP,
            win_length=STFT_WIN_LEN,
            lap_k=LAPLACIAN_K,
            global_mean=global_mean,
            global_std=global_std,
        )
        if val_ds_base is not None
        else None
    )

    test_ds = CrossSubjectLaplacianSpectrogramDataset(
        test_ds_base,
        fallback_coords=fallback_coords,
        n_fft=STFT_N_FFT,
        hop_length=STFT_HOP,
        win_length=STFT_WIN_LEN,
        lap_k=LAPLACIAN_K,
        global_mean=global_mean,
        global_std=global_std,
    )

    for i in range(min(len(train_ds_base), 2048)):
        try:
            _, _, meta = unpack_base_item(train_ds_base[i])
            train_subject_ids.add(infer_subject_id(meta))
        except Exception:
            pass

    train_loaders[task_name] = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_cross_subject_batch,
    )

    if val_ds is not None:
        val_loaders[task_name] = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
            collate_fn=collate_cross_subject_batch,
        )

    test_loaders[task_name] = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_cross_subject_batch,
    )

sample_batch = next(iter(train_loaders["speech"]))
print("Sample x_spec batch shape:", tuple(sample_batch["x_spec"].shape))
print("Sample coords batch shape:", tuple(sample_batch["coords"].shape))
print("Sample electrode_mask batch shape:", tuple(sample_batch["electrode_mask"].shape))
print("Sample n_electrodes:", sample_batch["n_electrodes"][:8].tolist())

if "speech" in val_loaders:
    val_batch = next(iter(val_loaders["speech"]))
    print("Val x_spec batch shape:", tuple(val_batch["x_spec"].shape))

num_subjects = max(SUBJECT_IDS) + 1
print("Estimated num_subjects:", num_subjects)

KeyboardInterrupt: 

In [ ]:
# =========================
# Init model
# =========================
model = LeaderboardMultiTaskModel(
    num_subjects=num_subjects,
    tasks=TASKS,
    dropout=0.25,
    max_electrodes=512,
).to(DEVICE)

criteria = {task: nn.BCEWithLogitsLoss() for task in TASKS}

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

In [ ]:
# =========================
# Label sanity checks (run once before training)
# =========================

CHECK_TASKS = ["speech", "delta_volume", "pitch", "gpt2_surprisal", "word_gap"]

for task_name in CHECK_TASKS:
    print(f"\nChecking labels for task: {task_name}")
    train_ds_base, eval_ds_base, fold_info = get_cross_subject_datasets(task_name, split_idx=0)

    # Extract up to 2000 labels from train and eval
    y_train = np.array([unpack_base_item(train_ds_base[i])[1].item()
                        for i in range(min(len(train_ds_base), 2000))])
    y_eval  = np.array([unpack_base_item(eval_ds_base[i])[1].item()
                        for i in range(min(len(eval_ds_base), 2000))])

    uniq_train = np.unique(y_train)
    uniq_eval  = np.unique(y_eval)

    print("  train unique:", uniq_train[:20], "n_unique:", len(uniq_train))
    print("  eval  unique:", uniq_eval[:20],  "n_unique:", len(uniq_eval))

for task_name in CHECK_TASKS:
    train_ds_base, eval_ds_base, _ = get_cross_subject_datasets(task_name, split_idx=0)
    y_train = np.array([unpack_base_item(train_ds_base[i])[1].item()
                        for i in range(min(len(train_ds_base), 2000))])
    uniq_train = np.unique(y_train)
    assert set(np.round(uniq_train, 6)).issubset({0.0, 1.0}), \
        f"{task_name} is not binary in training data: {uniq_train[:10]}"

In [ ]:
# =========================
# Train
# =========================
best_score = -np.inf
best_state_dict = None
epochs_without_improvement = 0
history_rows = []

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n========== CROSS-SUBJECT LEADERBOARD RUN Epoch {epoch:02d} ==========")

    epoch_train_losses = {}
    epoch_eval = {}

    task_order = ["speech", "delta_volume", "pitch", "gpt2_surprisal", "word_gap"]

    for task_name in task_order:
        model.train()
        total_loss = 0.0
        n_examples = 0

        for batch in train_loaders[task_name]:
            for key, value in batch.items():
                batch[key] = value.to(DEVICE)

            y = batch["y"]

            optimizer.zero_grad()

            preds = model(batch, task_name)
            raw_loss = criteria[task_name](preds, y)
            loss = model.weighted_loss(raw_loss, task_name)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += raw_loss.item() * y.size(0)
            n_examples += y.size(0)

        epoch_train_losses[task_name] = total_loss / max(n_examples, 1)

        eval_out = evaluate_task(model, val_loaders[task_name], criteria[task_name], DEVICE, task_name)
        epoch_eval[task_name] = eval_out

        print(
            f"{task_name:15s} | "
            f"train_loss={epoch_train_losses[task_name]:.4f} | "
            f"val_loss={eval_out['loss']:.4f} | "
            f"{eval_out['metric_name']}={eval_out['metric']:.4f} | "
            f"baseline={eval_out['baseline_metric']:.4f} | "
            f"log_var={model.log_vars[task_name].item():.4f}"
        )

        history_rows.append({
            "epoch": epoch,
            "task": task_name,
            "train_loss": epoch_train_losses[task_name],
            "val_loss": eval_out["loss"],
            "metric_name": eval_out["metric_name"],
            "metric": eval_out["metric"],
            "baseline_metric": eval_out["baseline_metric"],
            "log_var": model.log_vars[task_name].item(),
        })

    score = combined_score(epoch_eval)
    print(f"Combined val score: {score:.4f}")

    if score > best_score:
        best_score = score
        best_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        # Save best checkpoint to disk
        checkpoint = {
            "epoch": epoch,
            "best_score": best_score,
            "model_state_dict": best_state_dict,
            "optimizer_state_dict": optimizer.state_dict(),
            "config": {
                "TASKS": TASKS,
                "TEST_SUBJECT_ID": TEST_SUBJECT_ID,
                "TEST_TRIAL_ID": TEST_TRIAL_ID,
                "STFT_N_FFT": STFT_N_FFT,
                "STFT_HOP": STFT_HOP,
                "STFT_WIN_LEN": STFT_WIN_LEN,
                "LAPLACIAN_K": LAPLACIAN_K,
                "BATCH_SIZE": BATCH_SIZE,
                "BASE_LR": BASE_LR,
                "WEIGHT_DECAY": WEIGHT_DECAY,
            },
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"Saved new best checkpoint to {CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1

    print(f"Best combined val score so far: {best_score:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

history_df = pd.DataFrame(history_rows)
print("\nBest combined val score:", best_score)
display(history_df.tail(20))

In [ ]:
print("\nFinal TEST evaluation with best checkpoint:")
final_rows = []

for task_name in TASKS:
    out = evaluate_task(model, test_loaders[task_name], criteria[task_name], DEVICE, task_name)
    print(
        f"[FINAL TEST] {task_name:15s} | "
        f"loss={out['loss']:.4f} | "
        f"auroc={out['metric']:.4f}"
    )
    final_rows.append({
        "task": task_name,
        "test_loss": out["loss"],
        "test_auroc": out["metric"],
    })

final_test_df = pd.DataFrame(final_rows)
display(final_test_df)

In [ ]:
from pathlib import Path
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

outdir = Path("output")
outdir.mkdir(exist_ok=True)

hist = history_df.copy()
hist["auroc"] = hist["metric"]

combined = (
    hist.groupby("epoch", as_index=False)
        .apply(lambda g: pd.Series({
            "combined_score": float((g["metric"] - g["baseline_metric"]).mean())
        }))
        .reset_index(drop=True)
)

tasks = ["speech", "delta_volume", "pitch", "gpt2_surprisal", "word_gap"]

sns.set(style="whitegrid")

# 1) Validation AUROC by task
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=hist,
    x="epoch",
    y="metric",
    hue="task",
    marker="o"
)
plt.axhline(0.5, ls="--", color="grey", alpha=0.7)
plt.title("Val AUROC by task")
plt.xlabel("Epoch")
plt.ylabel("Val AUROC")
plt.tight_layout()
plt.savefig(outdir / "val_auroc_by_task.png", dpi=200)
plt.show()
plt.close()

# 2) Train vs val loss by task (facets)
loss_long = hist.melt(
    id_vars=["epoch", "task"],
    value_vars=["train_loss", "val_loss"],
    var_name="split",
    value_name="loss"
)

g = sns.FacetGrid(
    loss_long,
    col="task",
    col_wrap=3,
    sharex=False,
    sharey=False,
    height=3
)
g.map_dataframe(
    sns.lineplot,
    x="epoch",
    y="loss",
    hue="split",
    marker="o"
)
g.add_legend()
g.fig.suptitle("Train vs validation loss by task", y=1.03)
for ax in g.axes.flatten():
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
plt.tight_layout()
g.savefig(outdir / "loss_by_task.png", dpi=200)
plt.show()
plt.close()

# 3) Combined score trend
plt.figure(figsize=(6, 4))
sns.lineplot(
    data=combined,
    x="epoch",
    y="combined_score",
    marker="o"
)
plt.axhline(0.0, ls="--", color="grey", alpha=0.7)
plt.fill_between(
    combined["epoch"],
    combined["combined_score"],
    0.0,
    alpha=0.2
)
plt.title("Combined validation score")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig(outdir / "combined_score_trend.png", dpi=200)
plt.show()
plt.close()

# 4) Best val vs final test AUROC
best_val = (
    hist.groupby("task", as_index=False)["metric"]
        .max()
        .rename(columns={"metric": "best_val_auroc"})
)
comp = best_val.merge(
    final_test_df[["task", "test_auroc"]],
    on="task",
    how="left"
)

comp_long = comp.melt(
    id_vars="task",
    value_vars=["best_val_auroc", "test_auroc"],
    var_name="split",
    value_name="auroc"
)

plt.figure(figsize=(6, 4))
sns.barplot(
    data=comp_long,
    x="task",
    y="auroc",
    hue="split"
)
plt.axhline(0.5, ls="--", color="grey", alpha=0.7)
plt.title("Best validation vs final test AUROC")
plt.xlabel("Task")
plt.ylabel("AUROC")
plt.tight_layout()
plt.savefig(outdir / "val_test_compare.png", dpi=200)
plt.show()
plt.close()

# 5) Log-var trend
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=hist,
    x="epoch",
    y="log_var",
    hue="task",
    marker="o"
)
plt.title("Learned task log-variance")
plt.xlabel("Epoch")
plt.ylabel("Log var")
plt.tight_layout()
plt.savefig(outdir / "logvar_trend.png", dpi=200)
plt.show()
plt.close()